In [ ]:
# Create a VectorAssembler to combine multiple feature columns into a single vector column
# Spark ML models expect input features in a single vector format (not separate columns)

vector_assembler = VectorAssembler(
    inputCols=['rate_marriage', 'age', 'yrs_married', 'children', 'religious'],  # selected independent variables
    outputCol="featureVector"  # new column that will store all features as a vector
)

# After transformation, each row will have:
# featureVector = [rate_marriage, age, yrs_married, children, religious]

In [ ]:
# Initialize a Gradient Boosted Trees (GBT) classifier
# GBT is an ensemble method that builds trees sequentially,
# where each new tree tries to correct errors from previous ones (boosting concept)

gbt_classifier = GBTClassifier(
    labelCol="affairs",          # target variable (what we want to predict)
    featuresCol="featureVector", # input features (must be a vector column from VectorAssembler)
    predictionCol="prediction",  # output column storing predicted class (0 or 1)
)

# After training:
# model = gbt_classifier.fit(train)

# After prediction:
# result = model.transform(test)

# The result DataFrame will contain:
# - prediction → final predicted class
# - probability → probability of each class
# - rawPrediction → internal score before probability conversion

**Built model**

In [ ]:
# Pipeline chains multiple stages into one workflow:
# Step 1: Convert columns → feature vector
# Step 2: Train GBT model

df_pipeline = Pipeline(stages=[vector_assembler, gbt_classifier])

# ParamGridBuilder defines combinations of parameters to try
# This is used for model tuning (Grid Search)

# trick - When you use ( ) parentheses, Python automatically allows multi-line expressions without using '\'.
paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt_classifier.maxIter, [10, 20, 30]) # number of boosting iterations (trees)
    .addGrid(gbt_classifier.maxDepth, [2,3,5,6,8,9,10]) # tree depth (model complexity)
    .build()
)

# BinaryClassificationEvaluator evaluates model performance
# rawPrediction is used internally (before probability conversion)
# Default metric = areaUnderROC (AUC)

evaluator = BinaryClassificationEvaluator(
    rawPredictionCol='rawPrediction',
    labelCol='affairs'
)

# TrainValidationSplit:
# - Splits training data into train + validation (e.g., 80/20)
# - Trains models on train set
# - Evaluates on validation set
# - Selects best model

validator = TrainValidationSplit(
    estimator=df_pipeline,          # pipeline to train
    estimatorParamMaps=paramGrid,   # parameter combinations
    evaluator=evaluator,            # evaluation metric
    trainRatio=0.8                  # 80% train, 20% validation
)

# Train Model (Hyperparameter Tuning)
# This step:
# - tries all parameter combinations
# - performs cross-validation
# - selects the best model

validator_model = validator.fit(train)

# Make Predictions on Test Set
result = validator_model.transform(test)

# Inspect Results
# probability → probability of each class [P(0), P(1)]
# prediction → final predicted class (0 or 1)

result.select("affairs", "probability", "prediction").show(3, False)

**Evaluation**

- F1, AUC


In [ ]:
# Compute AUC (Area Under ROC Curve)
# This measures how well the model separates the two classes
# Higher AUC means better ranking performance
rf_auc = BinaryClassificationEvaluator(labelCol='affairs').evaluate(result)
print(rf_auc)

# Compute F1-score
# F1 combines precision and recall into one metric
# Useful when the classes are imbalanced
rf_f1 = MulticlassClassificationEvaluator(
    labelCol='affairs',
    metricName='f1'
).evaluate(result)
print(rf_f1)

Check Parameter
- Parameter
- ตัวแแปรสำคัญ

In [ ]:
# Retrieve the best model selected by CrossValidator

# bestModel stores the pipeline with the best hyperparameter combination
best_model = validator_model.bestModel


# Print the parameters of the trained GBT model
# best_model.stages[1] refers to the second stage in the pipeline
# stage 0 = VectorAssembler
# stage 1 = GBTClassifier model
# extractParamMap() shows the final parameter settings of the best model
pprint(best_model.stages[1].extractParamMap())


# Extract feature importances from the trained GBT model
# Get the trained GBT model from the pipeline
gbt_model = best_model.stages[1]

# Pair each feature name with its importance score
# gbt_model.featureImportances returns the contribution of each feature
feature_importance_list = list(
    zip(df.columns[:-1], gbt_model.featureImportances.toArray())
)

# Sort features from most important to least important
feature_importance_list.sort(key=lambda x: x[1], reverse=True)

# Print the ranked feature importance list
pprint(feature_importance_list)

Improve Model